### Installs and Imports

In [ ]:
!pip install -q  -U transformers
!pip install -q -U datasets
!pip install -q -U evaluate
!pip install -q tokenizers

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

In [ ]:
from datasets import Dataset, DatasetDict
import evaluate


In [ ]:
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


### Load Training split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
save_dir = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"

train_file = os.path.join(save_dir, "train_split.json")
test_file = os.path.join(save_dir, "test_split.json")

train_df = pd.read_json(train_file)
test_df = pd.read_json(test_file)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print(train_df.columns.tolist())

Train shape: (3143, 7)
Test shape: (795, 7)
['source', 'market', 'date', 'instruction', 'table_data', 'report', 'prompts']


In [ ]:
display(train_df.head(2))

,source,market,date,instruction,table_data,report,prompts
0,totalfarmmarketing,cattle,2021-12-02,Please act as an expert financial market analy...,Date Pro...,Cattle futures posted moderate to strong gains...,{'instruction': 'Please act as an expert finan...
1,totalfarmmarketing,cattle,2022-01-14,Please act as an expert financial market analy...,Date Pro...,"Cattle prices are trying to turn higher, as th...",{'instruction': 'Please act as an expert finan...


In [ ]:
print(train_df["report"].isna().sum())
print(test_df["report"].isna().sum())

print(train_df["report"].str.len().describe())

0
0
count    3143.000000
mean      536.570474
std       363.832138
min        12.000000
25%       271.000000
50%       451.000000
75%       690.000000
max      2243.000000
Name: report, dtype: float64


In [ ]:
# inspect the raw table format
print(train_df.iloc[0]["table_data"])

for i in range(3):
    print(f"\n===== EXAMPLE {i} =====")
    print(train_df.iloc[i]["table_data"][:3000])



      Date                                 Product Name Symbol   Open   High    Low  Close  Volume
2021-11-02  Live Cattle Future (front month) (December)   LEZ1 128.90 130.43 128.88 129.95 21275.0
2021-11-03  Live Cattle Future (front month) (December)   LEZ1 130.90 132.40 130.62 131.65 27572.0
2021-11-04  Live Cattle Future (front month) (December)   LEZ1 131.57 132.00 130.55 130.62 23398.0
2021-11-05  Live Cattle Future (front month) (December)   LEZ1 130.85 131.93 130.68 131.80 31050.0
2021-11-08  Live Cattle Future (front month) (December)   LEZ1 131.70 132.50 131.70 132.10 30902.0
2021-11-09  Live Cattle Future (front month) (December)   LEZ1 131.88 132.38 131.40 132.20 30503.0
2021-11-10  Live Cattle Future (front month) (December)   LEZ1 131.90 132.38 131.30 132.00 31903.0
2021-11-11  Live Cattle Future (front month) (December)   LEZ1 132.15 132.32 131.38 131.88 25671.0
2021-11-12  Live Cattle Future (front month) (December)   LEZ1 131.90 132.62 131.48 132.12 21703.0
2021-11-15

#### Convert data in long format - table linearization

In [ ]:
from io import StringIO

def linearize_table(table_str):
    # Convert the raw string to a DataFrame using fixed-width format
    df = pd.read_fwf(StringIO(table_str))

    # Build "col: value" pairs row by row
    rows = []
    for _, row in df.iterrows():
        parts = [f"{col}: {row[col]}" for col in df.columns]
        rows.append(", ".join(parts))

    # Join rows with separator
    return " | ".join(rows)



In [ ]:
sample = train_df.iloc[0]["table_data"]
print(linearize_table(sample)[:2000])

Date: 2021-11-02, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 128.9, High: 130.43, Low: 128.88, Close: 129.95, Volume: 21275.0 | Date: 2021-11-03, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 130.9, High: 132.4, Low: 130.62, Close: 131.65, Volume: 27572.0 | Date: 2021-11-04, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 131.57, High: 132.0, Low: 130.55, Close: 130.62, Volume: 23398.0 | Date: 2021-11-05, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 130.85, High: 131.93, Low: 130.68, Close: 131.8, Volume: 31050.0 | Date: 2021-11-08, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 131.7, High: 132.5, Low: 131.7, Close: 132.1, Volume: 30902.0 | Date: 2021-11-09, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 131.88, High: 132.38, Low: 131.4, Close: 132.2, Volume: 30503.0 | Date: 2021-11-10, Pro

In [ ]:
print(repr(sample[:300]))

'      Date                                 Product Name Symbol   Open   High    Low  Close  Volume\n2021-11-02  Live Cattle Future (front month) (December)   LEZ1 128.90 130.43 128.88 129.95 21275.0\n2021-11-03  Live Cattle Future (front month) (December)   LEZ1 130.90 132.40 130.62 131.65 27572.0\n202'


check the linearized table

In [ ]:
sample = train_df.iloc[0]["table_data"]

parsed = pd.read_fwf(StringIO(sample))

print(parsed.shape)
print(parsed.columns.tolist())
print(parsed.head())
print(parsed.tail())
print(parsed.isna().sum())

(132, 8)
['Date', 'Product Name', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']
         Date                                 Product Name Symbol    Open  \
0  2021-11-02  Live Cattle Future (front month) (December)   LEZ1  128.90   
1  2021-11-03  Live Cattle Future (front month) (December)   LEZ1  130.90   
2  2021-11-04  Live Cattle Future (front month) (December)   LEZ1  131.57   
3  2021-11-05  Live Cattle Future (front month) (December)   LEZ1  130.85   
4  2021-11-08  Live Cattle Future (front month) (December)   LEZ1  131.70   

     High     Low   Close   Volume  
0  130.43  128.88  129.95  21275.0  
1  132.40  130.62  131.65  27572.0  
2  132.00  130.55  130.62  23398.0  
3  131.93  130.68  131.80  31050.0  
4  132.50  131.70  132.10  30902.0  
           Date                                Product Name Symbol    Open  \
127  2021-11-26  Feeder Cattle Future (third month) (April)   GFJ2  167.50   
128  2021-11-29  Feeder Cattle Future (third month) (April)   GFJ2  169.8

By doing linearization, we are fine-tuning the model's weights on paired (input, output) examples. It's the serialized form of structured data, because the encoder needs a linear sequence of tokens as input.

We are doing serialization because we are using pretrained encoder-decoder T5 or BART. Since these models were pretrained on text, their embeddings and positional encodings are built for token sequences, so the format cannot be a table cell or row or column.

Loading model t5-small

In [ ]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [ ]:
#  encoder input
def build_encoder_input(row):

    linear_table = linearize_table(row["table_data"])

    return (
        "Generate a financial market report. "
        f"Instruction: {row['instruction']} "
        f"Financial data: {linear_table}"
    )

# apply to df
train_df["encoder_input"] = train_df.apply(build_encoder_input, axis=1)
test_df["encoder_input"] = test_df.apply(build_encoder_input, axis=1)
train_df["decoder_target"] = train_df["report"]
test_df["decoder_target"] = test_df["report"]

In [ ]:
#  check input output length
def token_length(text):
    return len(tokenizer(str(text),truncation=False)["input_ids"])

In [ ]:
# calculate length
train_df["input_length"] = train_df["encoder_input"].apply(token_length)
train_df["target_length"] = train_df["decoder_target"].apply(token_length)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7823 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
print(train_df["input_length"].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))
print(train_df["target_length"].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

print("Input > 512:",(train_df["input_length"] > 512).mean())
print("Target > 256:",(train_df["target_length"] > 256).mean())

count     3143.000000
mean     12118.510022
std      11707.769667
min       2793.000000
50%       7435.000000
75%      12133.500000
90%      35953.000000
95%      36746.000000
99%      37755.660000
max      38392.000000
Name: input_length, dtype: float64
count    3143.000000
mean      131.379574
std        86.968900
min         6.000000
50%       113.000000
75%       169.500000
90%       257.000000
95%       320.900000
99%       393.000000
max       560.000000
Name: target_length, dtype: float64
Input > 512: 1.0
Target > 256: 0.10054088450524976


In [ ]:
#  input lengths are too long
sample = train_df.iloc[0]
print("Raw table_data length (chars):", len(sample["table_data"]))
print(sample["table_data"][:2000])
print("...")
print("Number of rows in raw table_data:", sample["table_data"].count("\n"))

Raw table_data length (chars): 13166
      Date                                 Product Name Symbol   Open   High    Low  Close  Volume
2021-11-02  Live Cattle Future (front month) (December)   LEZ1 128.90 130.43 128.88 129.95 21275.0
2021-11-03  Live Cattle Future (front month) (December)   LEZ1 130.90 132.40 130.62 131.65 27572.0
2021-11-04  Live Cattle Future (front month) (December)   LEZ1 131.57 132.00 130.55 130.62 23398.0
2021-11-05  Live Cattle Future (front month) (December)   LEZ1 130.85 131.93 130.68 131.80 31050.0
2021-11-08  Live Cattle Future (front month) (December)   LEZ1 131.70 132.50 131.70 132.10 30902.0
2021-11-09  Live Cattle Future (front month) (December)   LEZ1 131.88 132.38 131.40 132.20 30503.0
2021-11-10  Live Cattle Future (front month) (December)   LEZ1 131.90 132.38 131.30 132.00 31903.0
2021-11-11  Live Cattle Future (front month) (December)   LEZ1 132.15 132.32 131.38 131.88 25671.0
2021-11-12  Live Cattle Future (front month) (December)   LEZ1 131.90 13

In [ ]:
# quick check: how many rows does the raw table actually have?
train_df["raw_table_rows"] = train_df["table_data"].apply(lambda x: len([l for l in x.split("\n") if l.strip()]))
print(train_df["raw_table_rows"].describe())

count    3143.000000
mean      183.038180
std       150.757429
min        51.000000
25%        70.000000
50%       133.000000
75%       190.000000
max       515.000000
Name: raw_table_rows, dtype: float64


The length inputs are too long, we need to shorten the length so that the tokenizer can work on max 512 tokens. This is same with T5 base/ T5 large models

## Redoing Linearized table to reduce input length



In [ ]:
from io import StringIO

def linearize_table_compact(table_str):

    table_df = pd.read_fwf(StringIO(str(table_str)))

    # Remove completely empty rows and columns
    table_df = table_df.dropna(how="all")
    table_df = table_df.dropna(axis=1, how="all")

    # Constant metadata
    metadata_parts = []

    for col in ["Product Name", "Symbol"]:
        if col in table_df.columns:
            values = table_df[col].dropna().unique()

            if len(values) > 0:
                metadata_parts.append(f"{col}: {values[0]}")

    # Keep only useful numerical/time-series columns
    keep_cols = [
        col for col in [
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]if col in table_df.columns]

    table_df = table_df[keep_cols]

    # Convert rows into compact text
    header = " | ".join(table_df.columns)

    rows = []

    for _, row in table_df.iterrows():

        values = []

        for value in row:

            if pd.isna(value):
                values.append("N/A")
            else:
                values.append(str(value))

        rows.append(" | ".join(values))

    table_text = "\n".join(rows)

    return "\n".join(metadata_parts) + "\n\n" + header + "\n" + table_text

In [ ]:
sample = train_df.iloc[0]["table_data"]
print(linearize_table(sample)[:2000])

Date: 2021-11-02, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 128.9, High: 130.43, Low: 128.88, Close: 129.95, Volume: 21275.0 | Date: 2021-11-03, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 130.9, High: 132.4, Low: 130.62, Close: 131.65, Volume: 27572.0 | Date: 2021-11-04, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 131.57, High: 132.0, Low: 130.55, Close: 130.62, Volume: 23398.0 | Date: 2021-11-05, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 130.85, High: 131.93, Low: 130.68, Close: 131.8, Volume: 31050.0 | Date: 2021-11-08, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 131.7, High: 132.5, Low: 131.7, Close: 132.1, Volume: 30902.0 | Date: 2021-11-09, Product Name: Live Cattle Future (front month) (December), Symbol: LEZ1, Open: 131.88, High: 132.38, Low: 131.4, Close: 132.2, Volume: 30503.0 | Date: 2021-11-10, Pro

In [ ]:
sample = train_df.iloc[0]["table_data"]

parsed = pd.read_fwf(StringIO(sample))

print(parsed.shape)
print(parsed.columns.tolist())
print(parsed.head())
print(parsed.tail())
print(parsed.isna().sum())

(132, 8)
['Date', 'Product Name', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']
         Date                                 Product Name Symbol    Open  \
0  2021-11-02  Live Cattle Future (front month) (December)   LEZ1  128.90   
1  2021-11-03  Live Cattle Future (front month) (December)   LEZ1  130.90   
2  2021-11-04  Live Cattle Future (front month) (December)   LEZ1  131.57   
3  2021-11-05  Live Cattle Future (front month) (December)   LEZ1  130.85   
4  2021-11-08  Live Cattle Future (front month) (December)   LEZ1  131.70   

     High     Low   Close   Volume  
0  130.43  128.88  129.95  21275.0  
1  132.40  130.62  131.65  27572.0  
2  132.00  130.55  130.62  23398.0  
3  131.93  130.68  131.80  31050.0  
4  132.50  131.70  132.10  30902.0  
           Date                                Product Name Symbol    Open  \
127  2021-11-26  Feeder Cattle Future (third month) (April)   GFJ2  167.50   
128  2021-11-29  Feeder Cattle Future (third month) (April)   GFJ2  169.8

The above function did not reduce the length.

In [ ]:
# recreate encoder
def build_encoder_input(row):

    linear_table = linearize_table_compact(row["table_data"])

    return (
        "Generate a financial market report.\n"
        f"Instruction: {row['instruction']}\n\n"
        f"Financial data:\n{linear_table}")

In [ ]:
train_df["encoder_input"] = train_df.apply(build_encoder_input,axis=1)

test_df["encoder_input"] = test_df.apply(build_encoder_input,axis=1)

In [ ]:
# recalculate
train_df["input_length"] = train_df["encoder_input"].apply(token_length)

train_df["target_length"] = train_df["decoder_target"].apply(token_length)

print(train_df["input_length"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

count     3143.000000
mean      4772.636335
std       4057.868174
min       1297.000000
50%       3339.000000
75%       4992.000000
90%      12945.600000
95%      13324.800000
99%      13702.000000
max      14113.000000
Name: input_length, dtype: float64


The above function did not help reduce the input length. The table contains multiple futures contracts in the same table_data. The first summary function incorrectly treated the entire table as one continuous time series. Each trading day has actually multiple contracts × observation. This needs to be fixed before training. We need to summarize each contract separately and calculate statistics within each Symbol. We should also remove blank row/columns, metadata, use correct datatypes etc.

In [ ]:
def summarize_market_table(table_str):

    table_df = pd.read_fwf(StringIO(str(table_str)))

    # Remove completely empty rows/columns
    table_df = table_df.dropna(how="all")
    table_df = table_df.dropna(axis=1, how="all")

    # Convert date
    if "Date" in table_df.columns:
        table_df["Date"] = pd.to_datetime(table_df["Date"],errors="coerce")

    # Convert numerical columns
    numeric_cols = [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
    for col in numeric_cols:
        if col in table_df.columns:
            table_df[col] = pd.to_numeric(table_df[col],errors="coerce")
    table_df = table_df.dropna(subset=["Close"]).reset_index(drop=True)

    if len(table_df) == 0:
        return ""

    summary = []

    # Metadata
    if "Product Name" in table_df.columns:
        product_name = table_df["Product Name"].dropna()

        if len(product_name) > 0:
            summary.append(f"Product: {product_name.iloc[0]}")

    if "Symbol" in table_df.columns:
        symbol = table_df["Symbol"].dropna()

        if len(symbol) > 0:
            summary.append(f"Symbol: {symbol.iloc[0]}")

    # Period information
    if "Date" in table_df.columns:

        start_date = table_df["Date"].min()
        end_date = table_df["Date"].max()

        summary.extend([
            f"Period start: {start_date.date()}",
            f"Period end: {end_date.date()}",
            f"Trading days: {len(table_df)}"
        ])

    # Price statistics
    close = table_df["Close"]
    start_close = close.iloc[0]
    end_close = close.iloc[-1]
    absolute_change = end_close - start_close
    percentage_change = ((end_close - start_close)/ start_close* 100)

    summary.extend([
        f"Starting close: {start_close:.2f}",
        f"Ending close: {end_close:.2f}",
        f"Period high: {table_df['High'].max():.2f}",
        f"Period low: {table_df['Low'].min():.2f}",
        f"Absolute price change: {absolute_change:.2f}",
        f"Percentage price change: {percentage_change:.2f}%"
    ])

    # Volatility
    daily_returns = close.pct_change().dropna()

    if len(daily_returns) > 0:
        volatility = daily_returns.std() * 100

        summary.append(f"Daily return volatility: {volatility:.2f}%")

        max_daily_gain = daily_returns.max() * 100
        max_daily_loss = daily_returns.min() * 100

        summary.extend([
            f"Maximum daily gain: {max_daily_gain:.2f}%",
            f"Maximum daily loss: {max_daily_loss:.2f}%"
        ])

    # Volume
    if "Volume" in table_df.columns:
        volume = table_df["Volume"].dropna()

        if len(volume) > 0:
            summary.extend([
                f"Average volume: {volume.mean():.0f}",
                f"Maximum volume: {volume.max():.0f}",
                f"Minimum volume: {volume.min():.0f}"
            ])

    # Trend
    if percentage_change > 0:
        trend = "Upward"
    elif percentage_change < 0:
        trend = "Downward"
    else:
        trend = "Flat"

    summary.append(f"Overall price trend: {trend}")

    return "\n".join(summary)

In [ ]:
summary = summarize_market_table(train_df.iloc[0]["table_data"])

print(summary)

Product: Live Cattle Future (front month) (December)
Symbol: LEZ1
Period start: 2021-11-02
Period end: 2021-12-02
Trading days: 132
Starting close: 129.95
Ending close: 170.90
Period high: 171.00
Period low: 128.88
Absolute price change: 40.95
Percentage price change: 31.51%
Daily return volatility: 1.25%
Maximum daily gain: 10.11%
Maximum daily loss: -4.69%
Average volume: 9696
Maximum volume: 31903
Minimum volume: 558
Overall price trend: Upward


Great! now the output looks smaller

In [ ]:
sample_df = pd.read_fwf(StringIO(str(train_df.iloc[0]["table_data"])))

sample_df["Date"] = pd.to_datetime(
    sample_df["Date"],
    errors="coerce"
)

print(sample_df.shape)
print(sample_df["Date"].min())
print(sample_df["Date"].max())
print(sample_df["Date"].nunique())

# print(sample_df[15:40])
print(sample_df.head())
print(sample_df.tail())

(132, 8)
2021-11-02 00:00:00
2021-12-02 00:00:00
22
        Date                                 Product Name Symbol    Open  \
0 2021-11-02  Live Cattle Future (front month) (December)   LEZ1  128.90   
1 2021-11-03  Live Cattle Future (front month) (December)   LEZ1  130.90   
2 2021-11-04  Live Cattle Future (front month) (December)   LEZ1  131.57   
3 2021-11-05  Live Cattle Future (front month) (December)   LEZ1  130.85   
4 2021-11-08  Live Cattle Future (front month) (December)   LEZ1  131.70   

     High     Low   Close   Volume  
0  130.43  128.88  129.95  21275.0  
1  132.40  130.62  131.65  27572.0  
2  132.00  130.55  130.62  23398.0  
3  131.93  130.68  131.80  31050.0  
4  132.50  131.70  132.10  30902.0  
          Date                                Product Name Symbol    Open  \
127 2021-11-26  Feeder Cattle Future (third month) (April)   GFJ2  167.50   
128 2021-11-29  Feeder Cattle Future (third month) (April)   GFJ2  169.88   
129 2021-11-30  Feeder Cattle Future (

In [ ]:
print(sample_df[["Date", "Close", "Volume"]].head(10))

print(sample_df[["Date", "Close", "Volume"]].tail(10))

        Date   Close   Volume
0 2021-11-02  129.95  21275.0
1 2021-11-03  131.65  27572.0
2 2021-11-04  130.62  23398.0
3 2021-11-05  131.80  31050.0
4 2021-11-08  132.10  30902.0
5 2021-11-09  132.20  30503.0
6 2021-11-10  132.00  31903.0
7 2021-11-11  131.88  25671.0
8 2021-11-12  132.12  21703.0
9 2021-11-15  131.78  14062.0
          Date   Close  Volume
122 2021-11-18  166.08  1267.0
123 2021-11-19  165.88  1283.0
124 2021-11-22  166.70   784.0
125 2021-11-23  168.02  1119.0
126 2021-11-24  169.80  1611.0
127 2021-11-26  169.88  1330.0
128 2021-11-29  168.58  2057.0
129 2021-11-30  168.82  2380.0
130 2021-12-01  170.45  1681.0
131 2021-12-02  170.90  1288.0


In [ ]:
print(sample_df["Date"].value_counts().head(10))

Date
2021-11-02    6
2021-11-03    6
2021-11-04    6
2021-11-05    6
2021-11-08    6
2021-11-09    6
2021-11-10    6
2021-11-11    6
2021-11-12    6
2021-11-15    6
Name: count, dtype: int64


In [ ]:
# Converting to .py file and continuing in model1_encoder-decoder.piynb
!jupyter nbconvert --to python "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/train_splitEDA.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/266/266_final_project/train_splitEDA.ipynb to python
[NbConvertApp] Writing 13198 bytes to /content/drive/MyDrive/Colab Notebooks/266/266_final_project/train_splitEDA.py


In [ ]:
!ls -l "/content/drive/MyDrive/Colab Notebooks/266/266_final_project"


total 488447
-rw------- 1 root root    130451 Jul 24 18:14 266_final_project_Text_analysis_pb.ipynb
-rw------- 1 root root      9247 Jul 20 23:18 266_git_workflow.ipynb
-rw------- 1 root root    147385 Jul 25 02:04 model1_encoder-decoder.ipynb
-rw------- 1 root root  51883332 Jul 24 17:53 test_split.json
-rw------- 1 root root 249532501 Jul 17 19:32 train.json
-rw------- 1 root root     88132 Jul 25 02:10 train_splitEDA.ipynb
-rw------- 1 root root     13199 Jul 25 02:10 train_splitEDA.py
-rw------- 1 root root 198363259 Jul 24 17:53 train_split.json
